# Queue Generation

Assumes the Neo4j graph has already been built and algorithms run in `spotify.ipynb`.

**Daily startup:** run all cells top to bottom. Takes ~30 seconds.

**To rebuild graph or re-run Louvain/Betweenness:** use `spotify.ipynb`.

In [154]:
import pandas as pd
import numpy as np
from neo4j import GraphDatabase

## Neo4j Connection

In [155]:
# neo4j connection and helper functions
driver = GraphDatabase.driver(
    'bolt://127.0.0.1:7687',
    auth=('neo4j', '205spotify'))

# convert results to pandas dataframe
def my_neo4j_run_query_pandas(query, **kwargs):
    with driver.session(database="ef66af10") as session:
        result = session.run(query, **kwargs)
        df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    return df

# run neo4j query
def neo4j_run(query, **kwargs):
    with driver.session(database="ef66af10") as session:
        return session.run(query, **kwargs)

## Graph Projection

In [ ]:
# re-project graph into GDS memory
# nodes and edges are persisted in neo4j - this just loads them into memory for algorithms
drop_proj_query = "CALL gds.graph.drop('song_graph', false) YIELD graphName"
neo4j_run(drop_proj_query)

proj_result = my_neo4j_run_query_pandas("""
CALL gds.graph.project(
    'song_graph',
    'Song',
    {
        SIMILAR_TO: {
            orientation: 'UNDIRECTED',
            properties: ['similarity', 'cost']
        }
    }
)
YIELD nodeCount, relationshipCount
RETURN nodeCount, relationshipCount
""")
print(proj_result)

### Cache Community Centroids

In [156]:
# pre-compute community centroids using server-side aggregation
# neo4j computes means across 81k songs and returns only 39 rows
centroid_df = my_neo4j_run_query_pandas("""
    MATCH (s:Song)
    WHERE s.community IS NOT NULL
    RETURN s.community        AS community,
           avg(s.danceability)     AS danceability,
           avg(s.energy)           AS energy,
           avg(s.loudness)         AS loudness,
           avg(s.speechiness)      AS speechiness,
           avg(s.acousticness)     AS acousticness,
           avg(s.instrumentalness) AS instrumentalness,
           avg(s.liveness)         AS liveness,
           avg(s.valence)          AS valence,
           avg(s.tempo)            AS tempo,
           avg(s.mode)             AS mode
""")

feature_cols = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence',
    'tempo', 'mode'
]

community_centroids = {
    row['community']: {col: row[col] for col in feature_cols}
    for _, row in centroid_df.iterrows()
}

print(f"Cached centroids for {len(community_centroids)} communities")

Cached centroids for 29 communities


## Queue Generation

### Identify Target Community
Calculate centroids of each community using feature vectors stored in Neo4j.
Find the community whose centroid is closest to the input community and set it as the target.

In [157]:
def find_target_community(input_community):
    input_centroid = np.array(list(community_centroids[input_community].values()))

    # rank all other communities by centroid distance
    distances = []
    for community, centroid_dict in community_centroids.items():
        if community == input_community:
            continue
        centroid = np.array(list(centroid_dict.values()))
        dist = np.linalg.norm(input_centroid - centroid)
        distances.append((community, dist))
    distances.sort(key=lambda x: x[1])
    ranked_communities = [c for c, _ in distances]

    # single batched query to get bridge counts for all communities at once
    all_bridge_counts = my_neo4j_run_query_pandas("""
        MATCH (bridge:Song)-[:SIMILAR_TO]-(a:Song),
              (bridge:Song)-[:SIMILAR_TO]-(b:Song)
        WHERE a.community = $input_community
          AND bridge.community IN [$input_community, a.community]
          AND bridge.betweenness IS NOT NULL
        RETURN DISTINCT b.community AS target_community,
               count(DISTINCT bridge) AS bridge_count
    """, input_community=int(input_community))

    valid_communities = set(
        all_bridge_counts[all_bridge_counts['bridge_count'] > 0]['target_community'].tolist()
    )

    # pick randomly from valid communities in top 5, expanding if needed
    rng = np.random.default_rng()
    for batch_size in [5, 10, 20, len(ranked_communities)]:
        valid = [c for c in ranked_communities[:batch_size] if c in valid_communities]
        if valid:
            return int(valid[rng.integers(0, len(valid))])

    raise ValueError(f'No reachable target community found from community {input_community}')

### Find Bridge Song

In [158]:
def find_bridge_song(input_community, target_community):
    # find top 3 highest-betweenness nodes connected to both input and target communities
    # bridge song must belong to either input or target community
    bridge_query = """
        MATCH (bridge:Song)-[:SIMILAR_TO]-(a:Song),
              (bridge:Song)-[:SIMILAR_TO]-(b:Song)
        WHERE a.community = $input_community
          AND b.community = $target_community
          AND bridge.community IN [$input_community, $target_community]
          AND bridge.betweenness IS NOT NULL
        RETURN DISTINCT
            bridge.id          AS track_id,
            bridge.track_name  AS track_name,
            bridge.genre       AS genre,
            bridge.community   AS community,
            bridge.betweenness AS betweenness
        ORDER BY betweenness DESC
        LIMIT 3
    """

    result = my_neo4j_run_query_pandas(
        bridge_query,
        input_community=int(input_community),
        target_community=int(target_community)
    )

    if result.empty:
        raise ValueError(f'No bridge song found between communities {input_community} and {target_community}')

    # pick randomly from top 3
    rng = np.random.default_rng()
    best = result.iloc[rng.integers(0, len(result))]
    return best['track_id'], int(best['community'])

### Pick Destination Song

In [159]:
def pick_destination(target_comm, bridge_id):
    # get all songs in target community excluding the bridge song
    target_query = """
    MATCH (s:Song)
    WHERE s.community = $target_community
    AND s.id <> $bridge_track_id
    RETURN
        s.id AS track_id, s.track_name AS track_name,
        s.genre AS genre, s.popularity AS popularity
    """

    target_output = my_neo4j_run_query_pandas(
        target_query,
        target_community=int(target_comm),
        bridge_track_id=bridge_id
    )

    rng = np.random.default_rng()
    target_track = target_output.iloc[rng.integers(0, len(target_output))]
    return target_track

### Dijkstra Shortest Path

In [160]:
def shortest_path(start_track_id, target_track_id):
    query = """
    MATCH (source:Song {id: $source_id}), (target:Song {id: $target_id})
    CALL gds.shortestPath.dijkstra.stream('song_graph',
        {
            sourceNode: source,
            targetNode: target,
            relationshipWeightProperty: 'cost'
        }
    )
    YIELD totalCost, nodeIds
    RETURN
        totalCost,
        [nodeId IN nodeIds | gds.util.asNode(nodeId).id] AS track_ids,
        [nodeId IN nodeIds | gds.util.asNode(nodeId).track_name] AS track_names,
        [nodeId IN nodeIds | gds.util.asNode(nodeId).artists] AS artists,
        [nodeId IN nodeIds | gds.util.asNode(nodeId).genre] AS genres,
        [nodeId IN nodeIds | gds.util.asNode(nodeId).community] AS communities
    """

    path_result = my_neo4j_run_query_pandas(
        query,
        source_id=start_track_id,
        target_id=target_track_id
    )

    if path_result.empty:
        raise ValueError(f'No path found between {start_track_id} and {target_track_id}')

    path = path_result.iloc[0]

    path_df = pd.DataFrame({
        'track_id':   path['track_ids'],
        'track_name': path['track_names'],
        'artist':     path['artists'],
        'genre':      path['genres'],
        'community':  path['communities']
    })
    path_df['total_cost'] = path['totalCost']

    return path_df

### Generate Queue

In [161]:
def generate_queue(input_track_id, queue_length=10, max_artist_appearances=2):
    # look up input song directly from neo4j
    song_result = my_neo4j_run_query_pandas("""
        MATCH (s:Song {id: $track_id})
        RETURN s.id AS track_id, s.track_name AS track_name,
               s.community AS community, s.artists AS artist
    """, track_id=input_track_id)

    if song_result.empty:
        raise ValueError(f'No song found with track_id: {input_track_id}')

    input_song = song_result.iloc[0]
    input_id = input_song['track_id']
    input_community = int(input_song['community'])

    print(f'Input song:       {input_song["track_name"]}')
    print(f'Input community:  {input_community}')

    target_community = find_target_community(input_community)
    print(f'Target community: {target_community}')

    bridge_id, bridge_community = find_bridge_song(input_community, target_community)
    print(f'Bridge community: {bridge_community}')

    dest_row = pick_destination(target_community, bridge_id)
    print(f'Destination song: {dest_row["track_name"]}')

    path1 = shortest_path(input_id, bridge_id)
    path2 = shortest_path(bridge_id, dest_row['track_id'])
    final_path = pd.concat([path1, path2.iloc[1:]], ignore_index=True)

    # deduplicate by track and name
    final_path = final_path.drop_duplicates(subset='track_id', keep='first')
    final_path = final_path.drop_duplicates(subset='track_name', keep='first')
    final_path = final_path.reset_index(drop=True)

    # --- artist repeat limiting ---
    artist_counts = final_path['artist'].value_counts()
    excess_artists = artist_counts[artist_counts > max_artist_appearances].index.tolist()

    for artist in excess_artists:
        artist_rows = final_path[final_path['artist'] == artist]
        to_replace = artist_rows.index[max_artist_appearances:]
        for idx in to_replace:
            current_id = final_path.loc[idx, 'track_id']
            used_ids = final_path['track_id'].tolist()
            replacement = my_neo4j_run_query_pandas("""
                MATCH (s:Song {id: $track_id})-[:SIMILAR_TO]->(n:Song)
                WHERE n.artists <> $artist
                  AND NOT n.id IN $used_ids
                RETURN n.id AS track_id, n.track_name AS track_name,
                       n.artists AS artist, n.genre AS genre,
                       n.community AS community
                ORDER BY n.betweenness IS NOT NULL DESC
                LIMIT 1
            """, track_id=current_id, artist=artist, used_ids=used_ids)
            if not replacement.empty:
                r = replacement.iloc[0]
                final_path.loc[idx, 'track_id'] = r['track_id']
                final_path.loc[idx, 'track_name'] = r['track_name']
                final_path.loc[idx, 'artist'] = r['artist']
                final_path.loc[idx, 'genre'] = r['genre']
                final_path.loc[idx, 'community'] = r['community']

    # --- enforce fixed queue length ---
    if len(final_path) > queue_length:
        final_path = final_path.iloc[:queue_length]
    elif len(final_path) < queue_length:
        used_ids = final_path['track_id'].tolist()
        while len(final_path) < queue_length:
            last_id = final_path.iloc[-1]['track_id']
            neighbors = my_neo4j_run_query_pandas("""
                MATCH (s:Song {id: $track_id})-[:SIMILAR_TO]->(n:Song)
                WHERE NOT n.id IN $used_ids
                RETURN n.id AS track_id, n.track_name AS track_name,
                       n.artists AS artist, n.genre AS genre,
                       n.community AS community
                ORDER BY n.betweenness IS NOT NULL DESC
                LIMIT 5
            """, track_id=last_id, used_ids=used_ids)
            if neighbors.empty:
                print('Warning: could not pad queue to full length')
                break
            rng = np.random.default_rng()
            next_song = neighbors.iloc[rng.integers(0, len(neighbors))]
            final_path = pd.concat([final_path, pd.DataFrame([{
                'track_id':   next_song['track_id'],
                'track_name': next_song['track_name'],
                'artist':     next_song['artist'],
                'genre':      next_song['genre'],
                'community':  next_song['community'],
                'total_cost': None
            }])], ignore_index=True)
            used_ids.append(next_song['track_id'])

    final_path = final_path.reset_index(drop=True)
    return final_path

## Run

In [162]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import threading

# mutable state container
state = {
    'selected_track_id': None,
    'timer': None,
    'selecting': False  # flag to suppress on_search when we set search_box.value programmatically
}

# --- widgets ---
search_box = widgets.Text(
    placeholder='Search for a song...',
    layout=widgets.Layout(width='480px')
)

results_box = widgets.VBox(
    [],
    layout=widgets.Layout(
        width='480px',
        border='1px solid #ccc',
        display='none'
    )
)

selected_label = widgets.HTML(value='')

generate_btn = widgets.Button(
    description='Generate Queue',
    button_style='primary',
    disabled=True,
    layout=widgets.Layout(width='160px')
)

output = widgets.Output()

# --- search logic ---
def do_search(search_term):
    results = my_neo4j_run_query_pandas("""
        MATCH (s:Song)
        WHERE toLower(s.track_name) CONTAINS toLower($search_term)
        RETURN s.id AS track_id, s.track_name AS track_name, s.artists AS artists
        ORDER BY s.popularity DESC
        LIMIT 8
    """, search_term=search_term)

    if results.empty:
        results_box.children = []
        results_box.layout.display = 'none'
        return

    buttons = []
    for _, row in results.iterrows():
        label = f"{row['track_name']} — {row['artists']}"
        track_id = row['track_id']
        btn = widgets.Button(
            description=label,
            layout=widgets.Layout(width='478px', height='36px'),
            style=widgets.ButtonStyle(button_color='#f8f8f8', font_size='13px')
        )
        def make_handler(tid, lbl):
            def on_result_click(b):
                state['selecting'] = True         # suppress on_search
                state['selected_track_id'] = tid
                search_box.value = lbl
                state['selecting'] = False        # re-enable on_search
                results_box.children = []
                results_box.layout.display = 'none'
                selected_label.value = f'<b>Selected:</b> {lbl}'
                generate_btn.disabled = False
            return on_result_click
        btn.on_click(make_handler(track_id, label))
        buttons.append(btn)

    results_box.children = buttons
    results_box.layout.display = 'block'

def on_search(change):
    if state['selecting']:
        return
    term = change['new']
    state['selected_track_id'] = None
    generate_btn.disabled = True
    selected_label.value = ''
    if len(term) < 2:
        results_box.children = []
        results_box.layout.display = 'none'
        return
    if state['timer'] is not None:
        state['timer'].cancel()
    state['timer'] = threading.Timer(0.3, do_search, args=[term])
    state['timer'].start()

def on_generate_click(b):
    with output:
        clear_output()
        if not state['selected_track_id']:
            print('Please select a song from the search results first.')
            return
        try:
            display(generate_queue(state['selected_track_id']))
        except ValueError as e:
            print(f'Error: {e}')

search_box.observe(on_search, names='value')
generate_btn.on_click(on_generate_click)

display(
    widgets.VBox([
        search_box,
        results_box,
        selected_label,
        generate_btn,
        output
    ])
)